# Data Wrangling Project – E-Wallet Payments & Transactions

**Context:** As a Data Analyst at an e-wallet company, analyze the following datasets to understand the state of payments and transactions:

- `payment_report.csv`: monthly payment volume of products
- `product.csv`: product information
- `transactions.csv`: transaction information

In [1]:
import pandas as pd
import numpy as np

DATA_DIR = "raws"  # on Google Colab: "/content/drive/MyDrive/PyThon_Project2"

payment_report = pd.read_csv(f"{DATA_DIR}/payment_report.csv")
products = pd.read_csv(f"{DATA_DIR}/product.csv")
transactions = pd.read_csv(f"{DATA_DIR}/transactions.csv")

## Part I: EDA - Explore Data Analysis

In [2]:
payment_enriched = payment_report.merge(products, on="product_id", how="left")
payment_enriched.head()

,report_month,payment_group,product_id,source_id,volume,category,team_own
0,2023-01,payment,12,45,624110375,PXXXXXT,ASD
1,2023-01,payment,17,45,335715113,PXXXXXB,ASD
2,2023-01,payment,18,45,737784466,PXXXXXB,ASD
3,2023-01,payment,19,45,120963069,PXXXXXM2,ASD
4,2023-01,payment,20,45,319653158,PXXXXXB,ASD


In [3]:
transactions.head()

,transaction_id,merchant_id,volume,transType,transStatus,sender_id,receiver_id,extra_info,timeStamp
0,3002692434,5,100000,24,1,10199794.0,199794.0,NaN,1682932054455
1,3002692437,305,20000,2,1,14022211.0,14022211.0,NaN,1682932054912
2,3001960110,7255,48605,22,1,NaN,10530940.0,NaN,1682932055000
3,3002680710,2270,1500000,2,1,10059206.0,59206.0,NaN,1682932055622
4,3002680713,2275,90000,2,1,10004711.0,4711.0,NaN,1682932056197


### Data types

In [4]:
payment_enriched.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 919 entries, 0 to 918
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   report_month   919 non-null    object
 1   payment_group  919 non-null    object
 2   product_id     919 non-null    int64 
 3   source_id      919 non-null    int64 
 4   volume         919 non-null    int64 
 5   category       897 non-null    object
 6   team_own       897 non-null    object
dtypes: int64(3), object(4)
memory usage: 50.4+ KB


In [5]:
transactions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1324002 entries, 0 to 1324001
Data columns (total 9 columns):
 #   Column          Non-Null Count    Dtype  
---  ------          --------------    -----  
 0   transaction_id  1324002 non-null  int64  
 1   merchant_id     1324002 non-null  int64  
 2   volume          1324002 non-null  int64  
 3   transType       1324002 non-null  int64  
 4   transStatus     1324002 non-null  int64  
 5   sender_id       1274943 non-null  float64
 6   receiver_id     1159207 non-null  float64
 7   extra_info      6095 non-null     object 
 8   timeStamp       1324002 non-null  int64  
dtypes: float64(2), int64(6), object(1)
memory usage: 90.9+ MB


In [6]:
payment_enriched["report_month"] = pd.to_datetime(payment_enriched["report_month"])
payment_enriched["report_month"].dtype

dtype('<M8[ns]')

`timeStamp` is Unix time in milliseconds (UTC). Convert it to Vietnam time (UTC+7).

In [7]:
transactions["timeStamp"] = (
    pd.to_datetime(transactions["timeStamp"], unit="ms")
    .dt.tz_localize("UTC")
    .dt.tz_convert("Asia/Ho_Chi_Minh")
    .dt.tz_localize(None)
)
transactions["timeStamp"].agg(["min", "max"])

min   2023-05-01 00:00:00.095
max   2023-05-07 23:59:59.495
Name: timeStamp, dtype: datetime64[ns]

In [8]:
transactions[["sender_id", "receiver_id"]] = transactions[["sender_id", "receiver_id"]].astype("Int64")
transactions[["sender_id", "receiver_id"]].dtypes

sender_id      Int64
receiver_id    Int64
dtype: object

### Missing data

In [9]:
payment_enriched.isna().sum()

report_month      0
payment_group     0
product_id        0
source_id         0
volume            0
category         22
team_own         22
dtype: int64

In [10]:
payment_enriched[payment_enriched["team_own"].isna()].groupby(["product_id", "payment_group"]).size()

product_id  payment_group
3           payment           1
1976        refund           17
10033       payment           4
dtype: int64

In [11]:
transactions.isna().sum()

transaction_id          0
merchant_id             0
volume                  0
transType               0
transStatus             0
sender_id           49059
receiver_id        164795
extra_info        1317907
timeStamp               0
dtype: int64

### Duplicates

In [12]:
payment_enriched.duplicated().sum()

np.int64(0)

In [13]:
payment_enriched[payment_enriched.duplicated(
    subset=["report_month", "payment_group", "product_id", "source_id"], keep=False)]

,report_month,payment_group,product_id,source_id,volume,category,team_own
194,2023-01-01,refund,1976,39,443387276,NaN,NaN
195,2023-01-01,refund,1976,39,111281678,NaN,NaN
409,2023-02-01,refund,1976,39,111500,NaN,NaN
412,2023-02-01,refund,1976,39,910819764,NaN,NaN
413,2023-02-01,refund,1976,39,2675894726,NaN,NaN
636,2023-03-01,refund,1976,39,1660931261,NaN,NaN
637,2023-03-01,refund,1976,39,4731275843,NaN,NaN
917,2023-04-01,refund,1976,39,1905435543,NaN,NaN
918,2023-04-01,refund,1976,39,3679922071,NaN,NaN


In [14]:
transactions.duplicated().sum()

np.int64(28)

In [15]:
transactions[transactions.duplicated(subset="transaction_id", keep=False)]

,transaction_id,merchant_id,volume,transType,transStatus,sender_id,receiver_id,extra_info,timeStamp
149734,3000871152,75,193000,2,1,17027174,<NA>,NaN,2023-05-02 07:50:17.608
149735,3000871152,75,193000,2,1,17027174,<NA>,NaN,2023-05-02 07:50:17.608
149738,3000871161,2270,50000,2,1,10009352,9352,NaN,2023-05-02 07:50:17.770
149739,3000871161,2270,50000,2,1,10009352,9352,NaN,2023-05-02 07:50:17.770
149752,3000687030,2250,165000,8,1,10074844,38548300,NaN,2023-05-02 07:50:23.437
149753,3000687030,2250,165000,8,1,10074844,38548300,NaN,2023-05-02 07:50:23.437
149757,3000871242,2270,300000,2,1,35013832,35013832,NaN,2023-05-02 07:50:25.523
149758,3000871242,2270,300000,2,1,35013832,35013832,NaN,2023-05-02 07:50:25.523
149761,3000871251,5,200,22,1,10028007,10529190,NaN,2023-05-02 07:50:26.225
149762,3000871251,5,200,22,1,10028007,10529190,NaN,2023-05-02 07:50:26.225


The duplicated rows share the same `transaction_id` and identical values in every column, so they are exact duplicates of the primary key and are removed.

In [16]:
transactions = transactions.drop_duplicates()
transactions["transaction_id"].duplicated().sum()

np.int64(0)

### Numerical values

In [17]:
payment_enriched.describe()

,report_month,product_id,source_id,volume
count,919,919.000000,919.000000,9.190000e+02
mean,2023-02-19 06:05:05.549510400,1192.517954,44.875952,1.978574e+08
min,2023-01-01 00:00:00,3.000000,37.000000,5.500000e+03
25%,2023-02-01 00:00:00,640.000000,45.000000,1.250000e+06
50%,2023-03-01 00:00:00,1059.000000,45.000000,7.982015e+06
75%,2023-04-01 00:00:00,1585.000000,45.000000,5.447599e+07
max,2023-04-01 00:00:00,15067.000000,45.000000,1.383171e+10
std,NaN,1293.463329,0.910995,8.367595e+08


In [18]:
transactions.describe()

,transaction_id,merchant_id,volume,transType,transStatus,sender_id,receiver_id,timeStamp
count,1.323974e+06,1.323974e+06,1.323974e+06,1.323974e+06,1.323974e+06,1274916.0,1159182.0,1323974
mean,3.002234e+09,2.460332e+03,2.388093e+05,6.979182e+00,-1.204636e+01,103392211.603019,208488236.021155,2023-05-03 23:13:08.211816192
min,3.000000e+09,5.000000e+00,1.000000e+00,2.000000e+00,-1.333000e+03,10000007.0,-63.0,2023-05-01 00:00:00.095000
25%,3.001121e+09,3.050000e+02,1.000000e+04,2.000000e+00,1.000000e+00,10056567.0,152670.0,2023-05-02 09:13:58.307000064
50%,3.002200e+09,2.250000e+03,3.000000e+04,2.000000e+00,1.000000e+00,10942004.0,7025130.0,2023-05-03 13:50:19.884499968
75%,3.003255e+09,2.270000e+03,1.000000e+05,8.000000e+00,1.000000e+00,21012663.0,31512376.0,2023-05-05 13:49:20.503000064
max,6.000066e+09,1.625250e+05,7.869148e+07,3.000000e+01,1.000000e+00,6993438823.0,21000000000.0,2023-05-07 23:59:59.495000
std,1.042617e+07,3.304293e+03,9.681107e+05,7.459665e+00,5.577851e+01,623425527.063003,928782507.772746,NaN


In [19]:
transactions[transactions["receiver_id"] < 0]

,transaction_id,merchant_id,volume,transType,transStatus,sender_id,receiver_id,extra_info,timeStamp
606160,3002176083,3955,134000,2,-217,10286027,-63,NaN,2023-05-07 13:28:17.965
606274,3002190501,3955,134000,2,-217,10286027,-63,NaN,2023-05-07 13:29:11.493
1126202,3002184327,3955,134000,2,-217,10286027,-63,NaN,2023-05-01 13:28:00.654


In [20]:
transactions["transStatus"].value_counts()

transStatus
 1       1248752
-217       67088
-358        2139
-351        1160
-357        1050
-68          961
-332         675
-333         313
-994         294
-54          270
-348         244
-200         205
-63          144
-344         118
-131         104
-356          81
-352          63
-354          61
-1333         46
-92           42
-361          39
-359          35
-308          28
-309          26
-998          18
-239           3
-129           3
-244           2
-161           2
-16            2
-87            2
-355           1
-340           1
-119           1
-267           1
Name: count, dtype: int64

In [21]:
(transactions["transStatus"] != 1).mean()

np.float64(0.05681531510437516)

### EDA Summary

**payment_enriched**

- Incorrect data types: `report_month` (object) → Next step: converted to datetime.
- Incorrect values: none (no negative or zero `volume`) → Next step: No action.
- Missing data: 22 rows in `category` and `team_own`, because `product_id` 3, 1976 and 10033 are not in `product.csv` (1976 accounts for all 17 refund rows) → Next step: No action, keep rows since their volume is valid.
- Duplicates: 0 exact duplicates. 5 rows repeat the key (`report_month`, `payment_group`, `product_id`, `source_id`) with different volumes, all refunds of product 1976 / source 39 → Next step: No action (multiple refunds in a month).

**transactions**

- Incorrect data types: `timeStamp` (Unix ms) → converted to datetime in UTC+7; `sender_id`, `receiver_id` (float because of NaN) → converted to nullable integer.
- Incorrect values: 3 rows with negative `receiver_id` (-63), all failed transactions → Next step: No action. `transStatus` ≠ 1 marks failed transactions (~5.7% of rows) → Next step: No action, keep for analysis.
- Missing data: 49,059 rows in `sender_id`, 164,795 rows in `receiver_id`, ~99.5% in `extra_info` → Next step: No action (optional fields depending on transaction type).
- Duplicates: PK `transaction_id`, 28 exact duplicate rows → Next step: Delete rows.

## Part II: Data Wrangling

### Using payment_report.csv & product.csv

**1. Top 3 product_ids with the highest volume.**

Only `payment_group = "payment"` is counted, since refunds are not payment volume.

In [22]:
top_products = (
    payment_enriched[payment_enriched["payment_group"] == "payment"]
    .groupby("product_id")["volume"].sum()
    .nlargest(3)
    .reset_index()
)
top_products

,product_id,volume
0,429,14667676567
1,372,13713658515
2,431,7821984747


**2. Given that 1 product_id is only owed by 1 team, are there any abnormal products against this rule?**

In [23]:
team_per_product = payment_enriched.groupby("product_id")["team_own"].nunique()
team_per_product[team_per_product > 1]

Series([], Name: team_own, dtype: int64)

→ No product is owned by more than one team, so there are no abnormal products.

**3. Find the team has had the lowest performance (lowest volume) since Q2.2023. Find the category that contributes the least to that team.**

The data ends in April 2023, so "since Q2.2023" covers April 2023 only. Rows without `team_own` are excluded.

In [24]:
q2 = payment_enriched[payment_enriched["report_month"] >= "2023-04-01"]

team_volume = q2.groupby("team_own")["volume"].sum().sort_values()
lowest_team = team_volume.idxmin()

category_volume = q2[q2["team_own"] == lowest_team].groupby("category")["volume"].sum().sort_values()
lowest_category = category_volume.idxmin()

print(team_volume, end="\n\n")
print(category_volume, end="\n\n")
print("lowest_team:", lowest_team)
print("lowest_category:", lowest_category)

team_own
APS       51141753
ASL    11284361730
ASD    31090428473
Name: volume, dtype: int64

category
PXXXXXE    25232438
PXXXXXS    25909315
Name: volume, dtype: int64

lowest_team: APS
lowest_category: PXXXXXE


**4. Find the contribution of source_ids of refund transactions (payment_group = 'refund'), what is the source_id with the highest contribution?**

In [25]:
refund = payment_enriched[payment_enriched["payment_group"] == "refund"]

refund_by_source = refund.groupby("source_id")["volume"].sum().sort_values(ascending=False).to_frame()
refund_by_source["contribution_pct"] = (refund_by_source["volume"] / refund_by_source["volume"].sum() * 100).round(2)

print("highest_source_id:", refund_by_source.index[0])
refund_by_source

highest_source_id: 38


,volume,contribution_pct
source_id,,
38,36527454759,59.11
39,16119059662,26.08
37,9151069226,14.81


### Using transactions.csv

**5. Define type of transactions ('transaction_type') for each row, given:**
- transType = 2 & merchant_id = 1205: Bank Transfer Transaction
- transType = 2 & merchant_id = 2260: Withdraw Money Transaction
- transType = 2 & merchant_id = 2270: Top Up Money Transaction
- transType = 2 & others merchant_id: Payment Transaction
- transType = 8, merchant_id = 2250: Transfer Money Transaction
- transType = 8 & others merchant_id: Split Bill Transaction
- Remained cases are invalid transactions

In [26]:
conditions = [
    (transactions["transType"] == 2) & (transactions["merchant_id"] == 1205),
    (transactions["transType"] == 2) & (transactions["merchant_id"] == 2260),
    (transactions["transType"] == 2) & (transactions["merchant_id"] == 2270),
    (transactions["transType"] == 2),
    (transactions["transType"] == 8) & (transactions["merchant_id"] == 2250),
    (transactions["transType"] == 8)
]

choices = [
    "Bank Transfer Transaction",
    "Withdraw Money Transaction",
    "Top Up Money Transaction",
    "Payment Transaction",
    "Transfer Money Transaction",
    "Split Bill Transaction"
]

transactions["transaction_type"] = np.select(conditions, choices, default="Invalid Transaction")

transactions["transaction_type"].value_counts()

transaction_type
Payment Transaction           398665
Transfer Money Transaction    341173
Top Up Money Transaction      290498
Invalid Transaction           220658
Bank Transfer Transaction      37879
Withdraw Money Transaction     33725
Split Bill Transaction          1376
Name: count, dtype: int64

**6. Of each transaction type (excluding invalid transactions): find the number of transactions, volume, senders and receivers.**

Senders and receivers are counted as unique IDs.

In [27]:
valid = transactions[transactions["transaction_type"] != "Invalid Transaction"]

result = valid.groupby("transaction_type").agg(
    num_transaction=("transaction_id", "count"),
    total_volume=("volume", "sum"),
    num_sender=("sender_id", "nunique"),
    num_receiver=("receiver_id", "nunique")
).sort_values("total_volume", ascending=False)

result

,num_transaction,total_volume,num_sender,num_receiver
transaction_type,,,,
Top Up Money Transaction,290498,108605618829,110409,110409
Payment Transaction,398665,71850608441,139583,113298
Bank Transfer Transaction,37879,50605806190,23156,9271
Transfer Money Transaction,341173,37032880492,39021,34585
Withdraw Money Transaction,33725,23418181420,24814,24814
Split Bill Transaction,1376,4901464,1323,572


Success rate by transaction type (`transStatus = 1`):

In [28]:
success_rate = (
    valid.assign(is_success=valid["transStatus"] == 1)
    .groupby("transaction_type")["is_success"].mean()
    .mul(100).round(2)
    .sort_values()
)
success_rate

transaction_type
Top Up Money Transaction       82.74
Payment Transaction            94.96
Bank Transfer Transaction      96.41
Withdraw Money Transaction     99.91
Transfer Money Transaction     99.95
Split Bill Transaction        100.00
Name: is_success, dtype: float64

## Conclusion

- **Products:** product 429 and 372 lead payment volume (Jan – Apr 2023). Product 1976 carries all refund volume but is missing from `product.csv`, so the product master data should be completed.
- **Teams:** each product belongs to one team. Since Q2.2023, APS has by far the lowest volume, with category PXXXXXE contributing the least (very close to PXXXXXS).
- **Refunds:** source 38 accounts for ~59% of refund volume and is the first place to investigate.
- **Transactions (1 – 7 May 2023):** Top Up has the highest volume, while Payment has the most transactions and users. ~16.7% of rows do not match any defined transaction type, and Top Up has the lowest success rate, which is worth investigating.